<a href="https://colab.research.google.com/github/CharlestonA/IIITM_Codes/blob/main/FCNN_Regression_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- SECTION 2: DATA PREPARATION AND PROBLEM SETUP ---

# Problem: Regression - Predict median house values in California districts.
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

# 2. Explore the dataset
print("--- Initial Data Exploration ---")
print(f"Dataset Shape: {df.shape}")
print("\nData Types:\n", df.dtypes)
print("\nBasic Statistics:\n", df.describe())

# 3. Identify input features and target variable
X = df.drop(columns=['MedHouseVal']) # Input features
y = df['MedHouseVal']                # Target variable

# 4. Handle of missing values
print(f"\nMissing values per column:\n{df.isnull().sum()}")
# (Note: California housing is usually clean, but this check is a pedagogical requirement)

In [ ]:
# 5. Perform preprocessing steps
# Feature scaling (Standardization) is critical for stable/faster training
# There are no categorical variables in this dataset, so One-Hot Encoding is not required here.

# 6. Split the dataset into training, validation, and test sets
# Explanation: Validation data is used for tuning and monitoring overfitting
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

# Scaling features based ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 7. Basic sanity checks before model building
print("\n--- Pre-Modeling Sanity Checks ---")
print(f"Train Shape: {X_train_scaled.shape}, Val Shape: {X_val_scaled.shape}, Test Shape: {X_test_scaled.shape}")
print(f"Mean of scaled training data (should be ~0): {X_train_scaled.mean():.4f}")
print(f"Std of scaled training data (should be ~1): {X_train_scaled.std():.4f}")

# Reshaping data to match neural network input format (Tensors)
def prepare_tensors(X_data, y_data):
    X_tensor = torch.tensor(X_data, dtype=torch.float32)
    y_tensor = torch.tensor(y_data.values, dtype=torch.float32).view(-1, 1)
    return X_tensor, y_tensor

X_train_t, y_train_t = prepare_tensors(X_train_scaled, y_train)
X_val_t, y_val_t = prepare_tensors(X_val_scaled, y_val)
X_test_t, y_test_t = prepare_tensors(X_test_scaled, y_test)

# --- SECTION 4: BUILDING FCFNN MODELS (REGRESSION) ---

class RegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(RegressionNet, self).__init__()
        # Input layer based on number of features
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1) # Output layer for continuous values
        self.relu = nn.ReLU() # Activation function for learning patterns

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.output(x)

model = RegressionNet(X_train_scaled.shape[1])
criterion = nn.MSELoss() # Suitable loss function for regression
optimizer = optim.Adam(model.parameters(), lr=0.01) # Optimization approach

# --- SECTION 4 & 5: TRAINING AND HYPERPARAMETER ANALYSIS ---

train_losses, val_losses = [], []

print("\n--- Starting Training Loop ---")
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward() # Backward propagation
    optimizer.step()

    # Validation phase to monitor performance and overfitting
    model.eval()
    with torch.no_grad():
        v_outputs = model(X_val_t)
        v_loss = criterion(v_outputs, y_val_t)

    train_losses.append(loss.item())
    val_losses.append(v_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/100] | Train Loss: {loss.item():.4f} | Val Loss: {v_loss.item():.4f}")

# --- SECTION 6: EVALUATION AND COMPARISON ---

model.eval()
with torch.no_grad():
    predictions = model(X_test_t)
    test_mse = criterion(predictions, y_test_t)
    print(f"\nFinal Test MSE: {test_mse.item():.4f}")

# Visualizing results to identify overfitting/underfitting patterns
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss Curves: Analyzing Learning Trends')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()